# Identifying Elaboration Targets

This notebook is designed to identify the targets of given elaboration sentences. Specifically, it extracts two types of targets:

1. **Target Phrase:** The specific phrase (anchor phrase) that the elaboration sentence explains or provides additional context for.
2. **Target Sentence:** The specific sentence (anchor sentence) obtained from the context text, directly clarified or elaborated upon by the given elaboration sentence.

For target extraction **the GPT-4o model** is employed. The GPT-4o System Card provides detailed insights into the model's capabilities and limitations [OpenAI et al., 2024](https://arxiv.org/abs/2410.21276).
 The extracted targets are then validated using BERTScore and matching techniques to ensure accuracy and relevance to ensure that the elaboration sentence itself does not appear within the identified target sentence or target phrase.

# Generation model

In [2]:
import os
#os.environ["OPENAI_API_KEY"] ="Paste your API key here"

In [3]:
from openai import OpenAI 
import os

MODEL="gpt-4o"
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Data

In [6]:
from dataset_utils import load_dataset_from_csv

ds_type = "c2s"
setting = "target-sent-target"

dataset = load_dataset_from_csv(ds_type, setting)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['doc_num', 'source_text', 'label_text', 'elaboration_sentence', 'contextual_specificity_rating', 'target_sentence_4o', 'target_sentence_target'],
        num_rows: 1046
    })
    validation: Dataset({
        features: ['doc_num', 'source_text', 'label_text', 'elaboration_sentence', 'contextual_specificity_rating', 'target_sentence_4o', 'target_sentence_target'],
        num_rows: 132
    })
    test: Dataset({
        features: ['doc_num', 'source_text', 'label_text', 'elaboration_sentence', 'contextual_specificity_rating', 'target_sentence_4o', 'target_sentence_target'],
        num_rows: 116
    })
})


# Prompt design

In [7]:
SYSTEM_PROMPT_SUBJECT = """You are an expert in identifying the subject of the provided explanation sentence based on the context text. 
If the subject of the explanation sentence is a **pronoun (e.g., "it," "they," "he," "she")**, determine what the pronoun refers to within the context.
The subject MUST be written as a concise phrase (not as a complete sentence), and it MUST be found in the context text (it does not need to appear in the explanation sentence itself).
"""

SYSTEM_PROMPT_TARGET = """You are an expert in identifying the target phrase in a given text that provided explanation sentence is clarifying or simplifying. 
Return the main phrase which explanation sentence is referring to.
Return the identified phrase from the CONTEXT TEXT, not the explanation sentence itself.
"""

SYSTEM_PROMPT_TARGET_SENT = """
You are an expert in identifying the sentence that the provided explanation sentence clarifies or refers to. 
Your task is to return the sentence from the CONTEXT TEXT, not the explanation sentence itself!
"""

SYSTEM_PROMPT_TARGET_SENT_AND_TARGET_PHRASE = """
You are an expert in identifying unclear or complex terms and concepts in a given text.
Your task is to:
1. Identify the sentence from the CONTEXT TEXT that the provided explanation sentence clarifies or refers to.
2. Specify the exact phrase within that sentence that is being clarified.
Return the identified sentence and the phrase from the CONTEXT TEXT, not the explanation sentence itself.
"""

**All prompts are formatted using the ChatML format.**

In [17]:
def format_subject_example(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_SUBJECT},
            {"role": "user", "content":  "Identify the subject of the following explanation sentence: '{}' within the given text: '{}'".format(
    example["elaboration_sentence"], example["source_text"])}, 
        ]
    }

def format_target_example(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_TARGET},
            {"role": "user", "content":  "Identify the target phrase of the following explanation sentence: '{}' within the given text: '{}'".format(
    example["elaboration_sentence"], example["target_sentence_4o"])},
        ]
    }

def format_target_sent_example(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_TARGET_SENT},
            {"role": "user", "content":  "Identify the target sentence that the following explanation: '{}' clarifies or refers to within the given text: '{}'".format(
    example["elaboration_sentence"], example["source_text"])},
        ]
    }

def format_target_sent_and_target_phrase_example(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_TARGET_SENT_AND_TARGET_PHRASE},
            {"role": "user", "content":  "Identify the target sentence and target phrase in the context text: '{}' that is being clarified by the explanation sentence: '{}'".format(
    example["source_text"], example["elaboration_sentence"])},
        ]
    }
    
formatted_train_dataset = dataset["train"].map(format_target_sent_and_target_phrase_example)
formatted_validation_dataset = dataset["validation"].map(format_target_sent_and_target_phrase_example)
formatted_test_dataset = dataset["test"].map(format_target_sent_and_target_phrase_example)

datasets = {"train":formatted_train_dataset, "validation":formatted_validation_dataset, "test":formatted_test_dataset}

**Example**

In [5]:
formatted_test_dataset[0]["messages"][1]

{'content': "Identify the target sentence and target phrase in the context text: 'New companies have come that need skilled workers with more education. New Haven youth want those jobs, but they do not have the education or the skills.' that is being clarified by the explanation sentence: 'Many do not have the money to get the training they need.'",
 'role': 'user'}

# Target extraction

For target extraction, structured output is utilized to ensure the proper format of the model's response.

In [22]:
from pydantic import BaseModel

class ExplanationTarget(BaseModel):
    #subject: str
    target_sentence: str
    target_phrase: str

## Exemplary outputs

In [10]:
import random
example = random.choice(dataset["test"])
print(example)

{'doc_num': 768, 'source_text': "But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings' hockey victory.", 'label_text': "But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings' hockey victory. Still, many people don't like the idea of appearing in some stranger's video on the Internet.", 'elaboration_sentence': "Still, many people don't like the idea of appearing in some stranger's video on the Internet.", 'contextual_specificity_rating': 1, 'target_sentence_4o': '"But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings\' hockey victory." ', 'target_sentence_target': "'drones mainly take pictures of nature'"}


### Subject

In [14]:
formatted_example = format_subject_example(example)
print(formatted_example["messages"],end="\n\n")

completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=formatted_example["messages"],
    response_format= ExplanationTarget,
)

response = completion.choices[0].message.parsed
print(response)

[{'role': 'system', 'content': 'You are an expert in identifying the subject of the provided explanation sentence based on the context text. \nIf the subject of the explanation sentence is a **pronoun (e.g., "it," "they," "he," "she")**, determine what the pronoun refers to within the context.\nThe subject MUST be written as a concise phrase (not as a complete sentence), and it MUST be found in the context text (it does not need to appear in the explanation sentence itself).\n'}, {'role': 'user', 'content': "Identify the subject of the following explanation sentence: 'Still, many people don't like the idea of appearing in some stranger's video on the Internet.' within the given text: 'But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings' hockey victory.'"}]

subject='drones'


### Target phrase

In [18]:
formatted_example = format_target_example(example)
print(formatted_example["messages"],end="\n\n")

completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=formatted_example["messages"],
    response_format= ExplanationTarget,
)

response = completion.choices[0].message.parsed
print(response)

[{'role': 'system', 'content': 'You are an expert in identifying the target phrase in a given text that provided explanation sentence is clarifying or simplifying. \nReturn the main phrase which explanation sentence is referring to.\nReturn the identified phrase from the CONTEXT TEXT, not the explanation sentence itself.\n'}, {'role': 'user', 'content': 'Identify the target phrase of the following explanation sentence: \'Still, many people don\'t like the idea of appearing in some stranger\'s video on the Internet.\' within the given text: \'"But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings\' hockey victory." \''}]

target_phrase="appearing in some stranger's video on the Internet"


### Target sentence

In [20]:
formatted_example = format_target_sent_example(example)
print(formatted_example["messages"],end="\n\n")

completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=formatted_example["messages"],
    response_format= ExplanationTarget,
)

response = completion.choices[0].message.parsed
print(response)

[{'role': 'system', 'content': '\nYou are an expert in identifying the sentence that the provided explanation sentence clarifies or refers to. \nYour task is to return the sentence from the CONTEXT TEXT, not the explanation sentence itself!\n'}, {'role': 'user', 'content': "Identify the target sentence that the following explanation: 'Still, many people don't like the idea of appearing in some stranger's video on the Internet.' clarifies or refers to within the given text: 'But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings' hockey victory.'"}]

target_sentence='But drones mainly take pictures of nature.'


### Both target sentence and target phrase

In [23]:
formatted_example = format_target_sent_and_target_phrase_example(example)
print(formatted_example["messages"],end="\n\n")

completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=formatted_example["messages"],
    response_format= ExplanationTarget,
)

response = completion.choices[0].message.parsed
print(response)

[{'role': 'system', 'content': '\nYou are an expert in identifying unclear or complex terms and concepts in a given text.\nYour task is to:\n1. Identify the sentence from the CONTEXT TEXT that the provided explanation sentence clarifies or refers to.\n2. Specify the exact phrase within that sentence that is being clarified.\nReturn the identified sentence and the phrase from the CONTEXT TEXT, not the explanation sentence itself.\n'}, {'role': 'user', 'content': "Identify the target sentence and target phrase in the context text: 'But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings' hockey victory.' that is being clarified by the explanation sentence: 'Still, many people don't like the idea of appearing in some stranger's video on the Internet.'"}]

target_sentence="But drones mainly take pictures of nature. And they record historic moments, like the celebration of the Kings' hockey victory." target_phrase='And they record histo

## Results dataframe

In [10]:
import pandas as pd
splits = ["train","validation","test"]

for split in splits:
    df_results = pd.DataFrame({
        "doc_num": datasets[split]["doc_num"],
        "source_text" : datasets[split]["source_text"],
        "elaboration_sentence":datasets[split]["elaboration_sentence"],
        "target_sentence_4o": datasets[split]["target_sentence"],
        "target_sentence_target":"",
    })
    df_results.to_csv(f"../data/elaborations/{split}_ds_{setting}_elab_targets.csv", index=False)
print(setting)

c2o


## Extraction

**Column_names** : "subject ","target_sentence_4o", "target_sentence_target"

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from pydantic import BaseModel

class ExplanationTarget(BaseModel):
    target_phrase: str

splits = ["train","validation","test"]

for split in splits:
    df_results = pd.read_csv(f"../data/elaborations/{split}_ds_{setting}_elab_targets.csv")
    column_name = "target_sentence_target" # 
    df_results[column_name] = df_results[column_name].fillna("")
    formatted_dataset = datasets[split]
    
    for idx, example in tqdm(enumerate(formatted_dataset),total=len(formatted_dataset)):
        if df_results.at[idx,column_name] == "":
            try:
                completion = client.beta.chat.completions.parse(model=MODEL,messages=example["messages"],response_format=ExplanationTarget)
                df_results.at[idx,column_name] = completion.choices[0].message.parsed
                df_results.to_csv(f"../data/elaborations/{split}_ds_{setting}_elab_targets.csv",index=False)
            except Exception as e:
                print(f"{e} for index {idx} in {split}")
                df_results.at[idx, column_name] = "filtered"

# Target validation

In [116]:
import pandas as pd 
import os

data_path = "../data/elaborations"

ds = "c2sp"
split = "test"

dfs ={
    "train":os.path.join(data_path,"train", f"train_ds_{ds}_elab_targets.csv"),
    "validation":os.path.join(data_path, "validation", f"validation_ds_{ds}_elab_targets.csv"),
    "test":os.path.join(data_path,"test", f"test_ds_{ds}_elab_targets.csv")
}

df_t = pd.read_csv(dfs[split])

## Extraction of model responses

In [4]:
import re

def extract_target_components(response):

    target_sentence = None
    target_phrase = None

    target_sentence, target_phrase = response.split("target_phrase=")
    target_sentence = target_sentence.split("target_sentence=")[1]

    return target_sentence, target_phrase

response = "target_sentence='He called the talk a \"fireside hangout.\"' target_phrase='\"fireside hangout\"'"
target_sentence, target_phrase = extract_target_components(response)
print(target_sentence)
print(target_phrase)

'He called the talk a "fireside hangout."' 
'"fireside hangout"'


In [11]:
df_t["target_sentence_4o"] = df_t["target_sentence_4o"].astype("str")
df_t["target_sentence_target"] = df_t["target_sentence_target"].astype("str")

for idx, row in df_t.iterrows():
    response = row["response"]  
    if response:
        try: 
            target_sentence, target_phrase = extract_target_components(response)
            df_t.at[idx, "target_sentence_4o"] = target_sentence
            df_t.at[idx, "target_sentence_target"] = target_phrase
        except Exception as e:
            df_t.at[idx, "target_sentence_4o"] = ""
            df_t.at[idx, "target_sentence_target"] = ""

df_t.isnull().sum()

doc_num                     0
source_text                 0
label_text                134
elaboration_sentence        0
response                    0
target_sentence_4o          0
target_sentence_target      0
dtype: int64

## Validation 

This section validates the targets identified by the GPT-4o model by comparing the elaboration sentence with either the target sentence or the target phrase. The goal is to ensure relevance between the elaboration and its corresponding target. Validation is performed using methods such as **BERTScore** and **exact matching** to confirm that the extracted targets appropriately relate to the elaboration sentence without directly duplicating its content. 

### BERTScore

All records with a similarity score higher than 0.6, as determined by BERTScore, are manually reviewed.

In [160]:
from tqdm.notebook import tqdm
from bert_score import BERTScorer

scorer = BERTScorer(model_type='bert-base-uncased',device='cuda:0')


for index, row in tqdm(df_t.iterrows(), total=len(df_t)):
    elaboration = row['target_sentence_target']
    target = row['target_sentence_4o']
    try:
        #  BERTScore for this pair
        P, R, F1 = scorer.score(
            cands=[target],  
            refs=[elaboration],              
        )
        
        df_t.at[index,"targets_bsf1"] = F1.mean().item()
    except Exception as e:
        print(index)

  0%|          | 0/116 [00:00<?, ?it/s]

In [ ]:
df_check = df_t[df_t["targets_bsf1"]>0.6]
for index, row in df_check.iterrows():
    print("ID:", index)
    print("Similarity score:", round(row["targets_bsf1"],3))
    print(row["source_text"], end="\n\n")
    print("Elaboration:",row["elaboration_sentence"])
    print("Target sent:",row["target_sentence_4o"], end="\n\n")
    print("Target phrase:",row["target_sentence_target"], end="\n\n")
    print("-"*120)

### Exact Matching 

Targets are validated through a series of checks:

- whether the elaboration sentence is not directly contained within the target sentence.
- whether the target phrase appears in the elaboration sentence, and if it does, checks whether it is also present in the context text.
- whether the target phrase is present in the target sentence itself.

In [117]:
import re
from tqdm.notebook import tqdm

def is_substring_in_sentence(target_phrase, target_sentence, elaboration_sentence=None):
    """
    Check if a normalized version of the target phrase is in the target sentence.
    """

    def normalize(text):
        # convert to lowercase
        text = text.lower()
        # remove punctuation
        text = re.sub(r'[^\w\s]', '', text)
        # remove extra spaces
        text = " ".join(text.split())
        return text

    normalized_phrase = normalize(target_phrase)
    normalized_sentence = normalize(target_sentence)
    if elaboration_sentence:
        normalized_elab_sentence = normalize(elaboration_sentence)
        return normalized_phrase in normalized_sentence and normalized_phrase in normalized_elab_sentence

    return normalized_phrase in normalized_sentence

for index, row in tqdm(df_t.iterrows(), total=len(df_t)):
    target = row['target_sentence_target']
    target_sent = row['target_sentence_4o']
    elab_sent = row['elaboration_sentence']
    try:
        df_t.at[index,"target_isin"] = is_substring_in_sentence(target, target_sent)
    except Exception as e:
        print(index)

  0%|          | 0/116 [00:00<?, ?it/s]

In [ ]:
df_check = df_t[df_t["target_isin"]==True]
print(len(df_check))
for index, row in df_check.iterrows():
    print("ID:", index)
    #print("Similarity score:", round(row["targets_bsf1"],3))
    print(row["source_text"], end="\n\n")
    print("Elaboration:",row["elaboration_sentence"])
    print("Target sent:",row["target_sentence_4o"], end="\n\n")
    print("Target phrase:",row["target_sentence_target"], end="\n\n")
    print("-"*120)